In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn import pipeline
from sklearn import preprocessing

In [3]:
#Loading hiPSC data for our 10 cell lines
lines_DE = ["Kolf2", "Kolf3", "Kucg2", "Letw5", "Podx1", "Qolg1", "Sojd3", "Wibj2", "Yoch6"]
TPMS_reps = pd.read_csv('TPMS_hiPSC.csv', index_col=0)
#Keep only the lines we've got
lines = '|'.join(lines_DE)
TPMS_reps_lines = TPMS_reps.loc[:, TPMS_reps.columns.str.contains(lines+"|geneid")]

clusters=pd.read_csv("Supplementary_File_1.csv", index_col=0)

In [4]:
#DEGs in hiPSCs for which H3K27me3 was predictive of transcription.
geneid_K27=clusters[(clusters.cluster=='K27') | (clusters.cluster=='K4&K27') | (clusters.cluster=='K4&K27&ATAC')].gene_id

In [5]:
#Genes missing in GRCh37, which appear in GRCh38
missing=['ENSG00000054598', 'ENSG00000229637', 'ENSG00000273604',
       'ENSG00000275023', 'ENSG00000279692']

In [14]:
# Keeping only H3K27me3-regulated genes
K27_RNA_vals=TPMS_reps_lines[TPMS_reps_lines.geneid.isin(geneid_K27)]

# Excluding genes not present in GRCh37
K27_RNA_vals=K27_RNA_vals[~K27_RNA_vals.geneid.isin(missing)]

In [7]:
# Did the cell lines differentiate into PGCLCs? For each RNA replicate, 
# although success was evaluated at the cell line level.
DP=[0.218,0.218, 0.282,0.282, 0.768, 0.768, 0.522, 0.522, 0.095, 0.095, 0.19, 0.19, 0.365, 0.365, 0.35, 0.35, 0.409, 0.409, 0.409]

In [17]:
#Preparing the data for model.

K27_RNA_vals=K27_RNA_vals.iloc[:,:len(DP)]
K27_RNA_valsT=K27_RNA_vals.transpose()

In [19]:
#Preparing the data for bootstraping

Xall=K27_RNA_valsT.to_numpy()
Yall=np.array(DP)

samplesX = tuple(Xall[:, i] for i in range(Xall.shape[1]))
samples = samplesX + (Yall,)

In [20]:
import numpy as np
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

def multivariate_sigmoid(x, *params):
    w = np.array(params[:-1])
    c = params[-1]
    
    linear_combination = np.dot(w, x) + c
    linear_combination = np.clip(linear_combination, -100, 100) #Limiting possible values to prevent overflow
    return (1/ (1 + np.exp(-linear_combination)))

# Scikit-Learn Wrapper Class       
class RidgeSigmoidRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, C=1.0, method='L-BFGS-B'):
        # alpha is our L2 penalty strength (like in sklearn.linear_model.Ridge)
        self.C = C
        self.method = method
        
    def fit(self, X, y):
        X_T = X.T 
        n_features = X.shape[1]
        
        initial_guess = [0.0] * n_features + [0.0]
        
        def objective_function(params):
            y_pred = multivariate_sigmoid(X_T, *params)
            mse = np.mean((y - y_pred)**2)
            
            # L2 Penalty on weights ONLY (which are all params except the last one 'c')
            weights = np.array(params[:-1])
            alpha = 1/self.C
            l2_penalty = alpha * np.sum(weights**2)
            
            return mse + l2_penalty
        
        result = minimize(
            objective_function, 
            initial_guess, 
            method=self.method
        )
        
        if not result.success:
            print(f"Warning: Optimization failed. {result.message}")
            
        self.coef_ = result.x
        return self
    
    def predict(self, X):
        if not hasattr(self, 'coef_'):
            raise ValueError("The estimator has not been fitted yet.")
        return multivariate_sigmoid(X.T, *self.coef_)


In [21]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression

# I need subsample.
# Outght to be a function to be called with a given training sample.
def stability_l2(samplex, sampley, n_inner_boots=100):
    pipeline_stab = Pipeline([
        ('scaler', MinMaxScaler()),     
        ('sigmoid', RidgeSigmoidRegressor(C=10))])  

    total_size=len(sampley)
    boots_size=int(np.floor(total_size*0.8))

    pos_freqs=np.zeros(samplex.shape[1])
    neg_freqs=np.zeros(samplex.shape[1])
    norm=0
    
    for i in range(n_inner_boots):
        indices=np.random.choice(range(total_size), size=boots_size, replace=False)

        clf = pipeline_stab.fit(samplex[indices,:], sampley[indices])

        pos_vars=clf['sigmoid'].coef_[:-1] > 0
        neg_vars=clf['sigmoid'].coef_[:-1] < 0
        pos_freqs[pos_vars]+=1
        neg_freqs[neg_vars]+=1

        norm+=1

    n_pos_vars=pos_freqs/norm
    n_neg_vars=neg_freqs/norm

    return n_pos_vars, n_neg_vars

In [22]:
from sklearn.metrics import r2_score
#More strict L2 regularisation post-variable selection
rpipe = Pipeline([
        ('scaler', MinMaxScaler()),     
        ('sigmoid', RidgeSigmoidRegressor(C=10))]) 

# Cross-validation out of bag bootstrap.
def rmse_reg(X_samples, Y_samples, indices_init):
    out_of_bag = ~np.isin(np.arange(len(Y_samples)), indices_init)

    if np.sum(out_of_bag)==0:
        return np.nan, np.nan
        
    X_test=Xall[out_of_bag,:]
    y_test=Yall[out_of_bag]

    pos, neg = stability_l2(X_samples, Y_samples)
    f_pos=pos>0.95
    f_neg=neg>0.95
    f_tot=f_pos+f_neg
    
    X_train_sel=X_samples[:,f_tot]
    X_test_sel=X_test[:,f_tot]
    cl2f = rpipe.fit(X_train_sel, Y_samples)
    
    y_pred=cl2f.predict(X_test_sel)
    r2sc=r2_score(y_test, y_pred)
    mad = np.mean(abs(y_pred-y_test))

    return r2sc, mad

In [23]:
n_boots=200
n_points = len(Yall)
rmse_dist=np.zeros(n_boots)
mad_dist=np.zeros(n_boots)

np.random.seed(24601)

for i in range(n_boots):
    ##Bootstrap resampling
    indices = np.random.choice(range(n_points), size=n_points, replace=True)
    x_resampled = Xall[indices,:]
    y_resampled = Yall[indices]

    #Model training and test with stability selection. Threshold = 0.6
    rmse, mad = rmse_reg(x_resampled, y_resampled, indices)
    rmse_dist[i] = rmse
    mad_dist[i] = mad

In [24]:
rmse_stats = [np.percentile(rmse_dist, 2.5), np.median(rmse_dist), np.percentile(rmse_dist, 97.5)]
rmse_stats

[np.float64(-2.4638089682622537),
 np.float64(0.10009623550207541),
 np.float64(0.6137863786517168)]

In [25]:
mad_stats = [np.percentile(mad_dist, 2.5), np.median(mad_dist), np.percentile(mad_dist, 97.5)]
mad_stats

[np.float64(0.07515878006087767),
 np.float64(0.1213049446507893),
 np.float64(0.2209522327091572)]